# exp054 Stage 2 v2 Inference (Kaggle CPU)

**Purpose**: Stage 2 v2 ckpt で test_soundscapes 推論 → submission.csv

**Input**:
- `maekeso/birdclef2026-exp054-stage2-effv2s` (Stage 2 v2 ckpt)
- birdclef-2026 competition (test_soundscapes auto-mounted)

**Output**: submission.csv with TopN N=1 PP (paper 256 spec)

**Pipeline**:
1. Load Stage 2 v2 model (EffNetV2-S + SED head)
2. Read each test_soundscape file (60s)
3. Chunk to 12 × 5s segments
4. Forward pass → sigmoid probabilities
5. TopN N=1 PP scaling
6. Build submission.csv

**Expected runtime**: 60-70 min (Kaggle CPU 90 min cap)
**Expected LB**: 0.82-0.88 (standalone)


In [ ]:
# Cell 1: imports + paths
import os, sys, json, time, re
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf
import librosa
import timm
from tqdm.auto import tqdm

# Paths
CKPT_CANDIDATES = [
    Path("/kaggle/input/birdclef2026-exp054-stage2-effv2s/stage2_v2_best.pth"),
    Path("/kaggle/input/datasets/maekeso/birdclef2026-exp054-stage2-effv2s/stage2_v2_best.pth"),
]
CKPT_PATH = next((p for p in CKPT_CANDIDATES if p.exists()), None)
assert CKPT_PATH is not None, f"Stage 2 v2 ckpt not found: {CKPT_CANDIDATES}"
print(f"CKPT: {CKPT_PATH}")

TEST_CANDIDATES = [
    Path("/kaggle/input/birdclef-2026/test_soundscapes"),
    Path("/kaggle/input/competitions/birdclef-2026/test_soundscapes"),
]
TEST_DIR = next((p for p in TEST_CANDIDATES if p.exists()), None)
print(f"TEST_DIR: {TEST_DIR}")

# Sample submission for column order
SAMPLE_SUB_CANDIDATES = [
    Path("/kaggle/input/birdclef-2026/sample_submission.csv"),
    Path("/kaggle/input/competitions/birdclef-2026/sample_submission.csv"),
]
SAMPLE_SUB = next((p for p in SAMPLE_SUB_CANDIDATES if p.exists()), None)
sample = pd.read_csv(SAMPLE_SUB)
print(f"Sample sub: {len(sample)} rows, {len(sample.columns)} cols (incl row_id)")

DEVICE = torch.device("cpu")
torch.set_num_threads(4)


In [ ]:
# Cell 2: BC2026 species list (234)
__BC2026_SPECIES = [
    ('Guyalna cuta', '1161364', 'Insecta', 1161364),
    ('Caiman yacare', '116570', 'Reptilia', 116570),
    ('Leptodactylus luctator', '1176823', 'Amphibia', 1176823),
    ('Adenomera guarani', '1491113', 'Amphibia', 1491113),
    ('Lysapsus limellum', '1595929', 'Amphibia', 1595929),
    ('Equus caballus', '209233', 'Mammalia', 209233),
    ('Leptodactylus syphax', '22930', 'Amphibia', 22930),
    ('Leptodactylus mystacinus', '22956', 'Amphibia', 22956),
    ('Leptodactylus podicipinus', '22961', 'Amphibia', 22961),
    ('Leptodactylus elenae', '22967', 'Amphibia', 22967),
    ('Leptodactylus fuscus', '22973', 'Amphibia', 22973),
    ('Leptodactylus labyrinthicus', '22983', 'Amphibia', 22983),
    ('Leptodactylus petersii', '22985', 'Amphibia', 22985),
    ('Physalaemus centralis', '23150', 'Amphibia', 23150),
    ('Physalaemus albifrons', '23154', 'Amphibia', 23154),
    ('Physalaemus albonotatus', '23158', 'Amphibia', 23158),
    ('Pseudopaludicola mystacalis', '23176', 'Amphibia', 23176),
    ('Phyllomedusa sauvagii', '23724', 'Amphibia', 23724),
    ('Scinax nasicus', '24279', 'Amphibia', 24279),
    ('Scinax fuscovarius', '24285', 'Amphibia', 24285),
    ('Scinax fuscomarginatus', '24287', 'Amphibia', 24287),
    ('Scinax acuminatus', '24321', 'Amphibia', 24321),
    ('Quesada gigas', '244024', 'Insecta', 244024),
    ('Chiasmocleis mehelyi', '25073', 'Amphibia', 25073),
    ('Elachistocleis bicolor', '25092', 'Amphibia', 25092),
    ('Dermatonotus muelleri', '25214', 'Amphibia', 25214),
    ('Physalaemus biligonigerus', '326272', 'Amphibia', 326272),
    ('Panthera onca', '41970', 'Mammalia', 41970),
    ('Alouatta caraya', '43435', 'Mammalia', 43435),
    ('Canis familiaris', '47144', 'Mammalia', 47144),
    ('Insect son01', '47158son01', 'Insecta', 47158),
    ('Insect son02', '47158son02', 'Insecta', 47158),
    ('Insect son03', '47158son03', 'Insecta', 47158),
    ('Insect son04', '47158son04', 'Insecta', 47158),
    ('Insect son05', '47158son05', 'Insecta', 47158),
    ('Insect son06', '47158son06', 'Insecta', 47158),
    ('Insect son07', '47158son07', 'Insecta', 47158),
    ('Insect son08', '47158son08', 'Insecta', 47158),
    ('Insect son09', '47158son09', 'Insecta', 47158),
    ('Insect son10', '47158son10', 'Insecta', 47158),
    ('Insect son11', '47158son11', 'Insecta', 47158),
    ('Insect son12', '47158son12', 'Insecta', 47158),
    ('Insect son13', '47158son13', 'Insecta', 47158),
    ('Insect son14', '47158son14', 'Insecta', 47158),
    ('Insect son15', '47158son15', 'Insecta', 47158),
    ('Insect son16', '47158son16', 'Insecta', 47158),
    ('Insect son17', '47158son17', 'Insecta', 47158),
    ('Insect son18', '47158son18', 'Insecta', 47158),
    ('Insect son19', '47158son19', 'Insecta', 47158),
    ('Insect son20', '47158son20', 'Insecta', 47158),
    ('Insect son21', '47158son21', 'Insecta', 47158),
    ('Insect son22', '47158son22', 'Insecta', 47158),
    ('Insect son23', '47158son23', 'Insecta', 47158),
    ('Insect son24', '47158son24', 'Insecta', 47158),
    ('Insect son25', '47158son25', 'Insecta', 47158),
    ('Physalaemus nattereri', '476521', 'Amphibia', 476521),
    ('Sapajus cay', '516975', 'Mammalia', 516975),
    ('Pithecopus azureus', '517063', 'Amphibia', 517063),
    ('Boana lundii', '555123', 'Amphibia', 555123),
    ('Boana punctata', '555145', 'Amphibia', 555145),
    ('Boana raniceps', '555146', 'Amphibia', 555146),
    ('Ameerega picta', '64898', 'Amphibia', 64898),
    ('Dendropsophus minutus', '65377', 'Amphibia', 65377),
    ('Dendropsophus nanus', '65380', 'Amphibia', 65380),
    ('Pseudis platensis', '66971', 'Amphibia', 66971),
    ('Rhinella diptycha', '67107', 'Amphibia', 67107),
    ('Trachycephalus typhonius', '67252', 'Amphibia', 67252),
    ('Leptodactylus macrosternum', '70711', 'Amphibia', 70711),
    ('Plecturocebus pallescens', '738183', 'Mammalia', 738183),
    ('Bos taurus', '74113', 'Mammalia', 74113),
    ('Mico melanurus', '74580', 'Mammalia', 74580),
    ('Prionacris erosa', '760266', 'Insecta', 760266),
    ('Hylophilus pectoralis', 'ashgre1', 'Aves', 17431),
    ('Mustelirallus albicollis', 'astcra1', 'Aves', 508907),
    ('Crax fasciolata', 'bafcur1', 'Aves', 2046),
    ('Micrastur ruficollis', 'baffal1', 'Aves', 4699),
    ('Coereba flaveola', 'banana', 'Aves', 10199),
    ('Thamnophilus doliatus', 'barant1', 'Aves', 15764),
    ('Procnias nudicollis', 'batbel1', 'Aves', 8854),
    ('Ara ararauna', 'baymac', 'Aves', 19018),
    ('Dendrocygna autumnalis', 'bbwduc', 'Aves', 6893),
    ('Microspingus melanoleucus', 'bcwfin2', 'Aves', 558564),
    ('Donacobius atricapilla', 'bkcdon', 'Aves', 116877),
    ('Aratinga nenday', 'bkhpar', 'Aves', 367562),
    ('Busarellus nigricollis', 'blchaw1', 'Aves', 5346),
    ('Spizaetus tyrannus', 'blheag1', 'Aves', 5291),
    ('Tityra cayana', 'blttit1', 'Aves', 8830),
    ('Myiarchus tyrannulus', 'bncfly', 'Aves', 16016),
    ('Megarynchus pitangua', 'bobfly1', 'Aves', 16737),
    ('Progne tapera', 'brcmar1', 'Aves', 11870),
    ('Tyto furcata', 'brnowl', 'Aves', 1578502),
    ('Momotus momota', 'bucmot4', 'Aves', 204447),
    ('Thectocercus acuticaudatus', 'bucpar', 'Aves', 367564),
    ('Amazona aestiva', 'bufpar', 'Aves', 18978),
    ('Theristicus caudatus', 'bunibi1', 'Aves', 3766),
    ('Athene cunicularia', 'burowl', 'Aves', 19975),
    ('Colaptes campestris', 'camfli1', 'Aves', 18262),
    ('Ortalis canicollis', 'chacha1', 'Aves', 2088),
    ('Mimus saturninus', 'chbmoc1', 'Aves', 14878),
    ('Gnorimopsar chopi', 'chobla1', 'Aves', 10723),
    ('Conirostrum speciosum', 'chvcon1', 'Aves', 10013),
    ('Synallaxis hypospodia', 'cibspi1', 'Aves', 10921),
    ('Micrastur semitorquatus', 'coffal1', 'Aves', 4698),
    ('Nyctidromus albicollis', 'compau', 'Aves', 19627),
    ('Nyctibius griseus', 'compot1', 'Aves', 19667),
    ('Turdus amaurochalinus', 'crbthr1', 'Aves', 12710),
    ('Pachyramphus validus', 'crebec1', 'Aves', 8681),
    ('Taoniscus nanus', 'dwatin1', 'Aves', 20714),
    ('Icterus pyrrhopterus', 'epaori4', 'Aves', 72954),
    ('Lathrotriccus euleri', 'eulfly1', 'Aves', 17231),
    ('Cantorchilus guarayanus', 'fabwre1', 'Aves', 144880),
    ('Glaucidium brasilianum', 'fepowl', 'Aves', 19822),
    ('Machaeropterus pyrocephalus', 'ficman1', 'Aves', 14344),
    ('Myiothlypis flaveola', 'flawar1', 'Aves', 201223),
    ('Tyrannus savana', 'fotfly', 'Aves', 16793),
    ('Cnemotriccus fuscatus', 'fusfly1', 'Aves', 17076),
    ('Hylocharis chrysura', 'gilhum1', 'Aves', 5979),
    ('Aramides ypecaha', 'giwrai1', 'Aves', 460),
    ('Chionomesa fimbriata', 'glteme1', 'Aves', 1289639),
    ('Saltator coerulescens', 'grasal3', 'Aves', 9850),
    ('Crotophaga major', 'greani1', 'Aves', 1970),
    ('Taraba major', 'greant1', 'Aves', 15957),
    ('Myiopagis viridicata', 'greela', 'Aves', 16892),
    ('Pitangus sulphuratus', 'grekis', 'Aves', 16956),
    ('Nyctibius grandis', 'grepot1', 'Aves', 19680),
    ('Phacellodomus ruber', 'gretho2', 'Aves', 11632),
    ('Tringa melanoleuca', 'greyel', 'Aves', 3892),
    ('Leptotila rufaxilla', 'grfdov1', 'Aves', 3302),
    ('Eucometis penicillata', 'grhtan1', 'Aves', 10698),
    ('Aramides cajaneus', 'gycwor1', 'Aves', 513889),
    ('Anhima cornuta', 'horscr1', 'Aves', 6908),
    ('Passer domesticus', 'houspa', 'Aves', 13858),
    ('Anodorhynchus hyacinthinus', 'hyamac1', 'Aves', 18938),
    ('Elaenia spectabilis', 'larela1', 'Aves', 16734),
    ('Elaenia chiriquensis', 'lesela1', 'Aves', 578460),
    ('Emberizoides ypiranganus', 'lesgrf1', 'Aves', 10555),
    ('Aramus guarauna', 'limpki', 'Aves', 7),
    ('Dryocopus lineatus', 'linwoo1', 'Aves', 17858),
    ('Coccycua minuta', 'litcuc2', 'Aves', 72740),
    ('Setopagis parvula', 'litnig1', 'Aves', 367507),
    ('Pyrrhura frontalis', 'mabpar', 'Aves', 19162),
    ('Cercomacra melanaria', 'magant1', 'Aves', 15737),
    ('Cissopis leverianus', 'magtan2', 'Aves', 72727),
    ('Polioptila dumicola', 'masgna1', 'Aves', 7509),
    ('Chordeiles nacunda', 'nacnig1', 'Aves', 19661),
    ('Rufirallus schomburgkii', 'ocecra1', 'Aves', 1506288),
    ('Sittasomus griseicapillus', 'oliwoo1', 'Aves', 11511),
    ('Icterus croconotus', 'orbtro3', 'Aves', 62564),
    ('Amazona amazonica', 'orwpar', 'Aves', 18982),
    ('Pandion haliaetus', 'osprey', 'Aves', 116999),
    ('Synallaxis albescens', 'pabspi1', 'Aves', 10999),
    ('Furnarius leucopus', 'palhor3', 'Aves', 11281),
    ('Thraupis palmarum', 'paltan1', 'Aves', 10297),
    ('Dromococcyx phasianellus', 'phecuc1', 'Aves', 1982),
    ('Patagioenas picazuro', 'picpig2', 'Aves', 3102),
    ('Legatus leucophaius', 'pirfly1', 'Aves', 17312),
    ('Thamnophilus pelzelni', 'plasla1', 'Aves', 73493),
    ('Inezia inornata', 'platyr1', 'Aves', 16344),
    ('Cyanocorax chrysops', 'plcjay1', 'Aves', 8484),
    ('Theristicus caerulescens', 'pluibi1', 'Aves', 3768),
    ('Cyanocorax cyanomelas', 'purjay1', 'Aves', 8483),
    ('Hemitriccus margaritaceiventer', 'pvttyr1', 'Aves', 16273),
    ('Ara chloropterus', 'ragmac1', 'Aves', 19016),
    ('Campylorhamphus trochilirostris', 'rebscy1', 'Aves', 11201),
    ('Coryphospingus cucullatus', 'recfin1', 'Aves', 10310),
    ('Gallus gallus', 'redjun', 'Aves', 882),
    ('Cariama cristata', 'relser1', 'Aves', 14),
    ('Megaceryle torquata', 'rinkin1', 'Aves', 2552),
    ('Myiothlypis rivularis', 'rivwar1', 'Aves', 145267),
    ('Rupornis magnirostris', 'roahaw', 'Aves', 201041),
    ('Turdus rufiventris', 'rubthr1', 'Aves', 12738),
    ('Pseudoseisura unirufa', 'rufcac2', 'Aves', 11718),
    ('Casiornis rufus', 'rufcas2', 'Aves', 17102),
    ('Conopophaga lineata', 'rufgna3', 'Aves', 578313),
    ('Furnarius rufus', 'rufhor2', 'Aves', 11275),
    ('Antrostomus rufus', 'rufnig1', 'Aves', 201066),
    ('Phacellodomus rufifrons', 'ruftho1', 'Aves', 11624),
    ('Poecilotriccus latirostris', 'ruftof1', 'Aves', 17026),
    ('Myiozetetes cayanensis', 'rumfly1', 'Aves', 16833),
    ('Tigrisoma lineatum', 'ruther1', 'Aves', 5048),
    ('Galbula ruficauda', 'rutjac1', 'Aves', 1468),
    ('Arremon flavirostris', 'sabspa1', 'Aves', 10064),
    ('Sicalis flaveola', 'saffin', 'Aves', 9864),
    ('Thraupis sayaca', 'saytan1', 'Aves', 10293),
    ('Columbina squammata', 'scadov1', 'Aves', 3564),
    ('Pionus maximiliani', 'schpar1', 'Aves', 19094),
    ('Phaethornis eurynome', 'scther1', 'Aves', 5622),
    ('Myiarchus ferox', 'shcfly1', 'Aves', 16006),
    ('Accipiter striatus', 'shshaw', 'Aves', 5097),
    ('Lurocalis semitorquatus', 'shtnig1', 'Aves', 19645),
    ('Ramphocelus carbo', 'sibtan2', 'Aves', 10056),
    ('Crotophaga ani', 'smbani', 'Aves', 1971),
    ('Crypturellus parvirostris', 'smbtin1', 'Aves', 20570),
    ('Cacicus solitarius', 'sobcac1', 'Aves', 10365),
    ('Camptostoma obsoletum', 'sobtyr1', 'Aves', 16972),
    ('Myiozetetes similis', 'socfly1', 'Aves', 16842),
    ('Synallaxis frontalis', 'sofspi1', 'Aves', 10996),
    ('Corythopis delalandi', 'souant1', 'Aves', 17264),
    ('Vanellus chilensis', 'soulap1', 'Aves', 4867),
    ('Chauna torquata', 'souscr1', 'Aves', 6910),
    ('Hypoedaleus guttatus', 'spbant3', 'Aves', 15959),
    ('Synallaxis spixi', 'spispi1', 'Aves', 10915),
    ('Antiurus maculicaudus', 'sptnig1', 'Aves', 1584760),
    ('Piaya cayana', 'squcuc1', 'Aves', 1758),
    ('Dendroplex picus', 'stbwoo2', 'Aves', 72806),
    ('Tapera naevia', 'strcuc1', 'Aves', 1989),
    ('Butorides striata', 'strher2', 'Aves', 62528),
    ('Asio clamator', 'strowl1', 'Aves', 558468),
    ('Eupetomena macroura', 'swthum1', 'Aves', 6065),
    ('Chiroxiphia caudata', 'swtman1', 'Aves', 14306),
    ('Crypturellus tataupa', 'tattin1', 'Aves', 20587),
    ('Campylorhynchus turdinus', 'thlwre1', 'Aves', 7480),
    ('Ramphastos toco', 'toctou1', 'Aves', 18793),
    ('Tyrannus melancholicus', 'trokin', 'Aves', 16787),
    ('Megascops choliba', 'trsowl', 'Aves', 19788),
    ('Crypturellus undulatus', 'undtin1', 'Aves', 20592),
    ('Thamnophilus caerulescens', 'varant1', 'Aves', 15757),
    ('Jacana jacana', 'watjac1', 'Aves', 4580),
    ('Pyriglena maura', 'wesfie1', 'Aves', 1286886),
    ('Dendrocygna viduata', 'wfwduc1', 'Aves', 6898),
    ('Biatas nigropectus', 'whbant2', 'Aves', 15902),
    ('Myiothlypis leucoblephara', 'whbwar2', 'Aves', 201224),
    ('Melanerpes candidus', 'whiwoo1', 'Aves', 18183),
    ('Synallaxis albilora', 'whlspi1', 'Aves', 10992),
    ('Cyanocorax cyanopogon', 'whnjay1', 'Aves', 8469),
    ('Leptotila verreauxi', 'whtdov', 'Aves', 3280),
    ('Picumnus albosquamatus', 'whwpic1', 'Aves', 17786),
    ('Caracara plancus', 'y00678', 'Aves', 4715),
    ('Paroaria capitata', 'yebcar', 'Aves', 10257),
    ('Elaenia flavogaster', 'yebela1', 'Aves', 16714),
    ('Primolius auricollis', 'yecmac', 'Aves', 73272),
    ('Brotogeris chiriri', 'yecpar', 'Aves', 19215),
    ('Daptrius chimachima', 'yehcar1', 'Aves', 1432779),
    ('Tolmomyias sulphurescens', 'yeofly1', 'Aves', 16567),
]
species_df = pd.DataFrame(__BC2026_SPECIES, columns=["scientific_name", "primary_label", "class_name", "inat_taxon_id"])
PRIMARY_LABELS = species_df["primary_label"].tolist()
N_CLASSES = len(PRIMARY_LABELS)
print(f"BC2026: {N_CLASSES} species")


In [ ]:
# Cell 3: Model definition + ckpt load
SR = 32000
CHUNK_LEN = SR * 5  # 5s
N_WINDOWS = 12      # 60s / 5s
N_MELS = 128

class MelExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=SR, n_fft=2048, hop_length=512,
            n_mels=N_MELS, f_min=20, f_max=16000,
        )
        self.db = torchaudio.transforms.AmplitudeToDB(top_db=80.0)
    def forward(self, wav):
        mel = self.mel(wav); mel = self.db(mel)
        mel = torch.clamp(mel, -80.0, 0.0); mel = (mel + 40.0) / 40.0
        return mel

class SEDHead(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.att = nn.Linear(in_dim, n_classes)
        self.cla = nn.Linear(in_dim, n_classes)
    def forward(self, x):
        att = torch.tanh(self.att(x)); cla = self.cla(x)
        norm_att = F.softmax(att, dim=1)
        return (norm_att * cla).sum(dim=1)

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            "tf_efficientnetv2_s.in21k_ft_in1k", pretrained=False, in_chans=3,
            num_classes=0, global_pool="",
            drop_path_rate=0.0,    # eval: no stochastic depth
        )
        self.head = SEDHead(self.backbone.num_features, N_CLASSES)
    def forward(self, mel):
        x = mel.unsqueeze(1).repeat(1, 3, 1, 1)
        feat = self.backbone(x).mean(dim=2).transpose(1, 2)
        return self.head(feat)

# Load ckpt
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
model = Model()
model.load_state_dict(ckpt["state_dict"], strict=True)
model.eval()
mel_extractor = MelExtractor()
print(f"Stage 2 v2 ckpt loaded:")
print(f"  val_ns22: {ckpt.get('val_ns22', '?')}")
print(f"  val_macro: {ckpt.get('val_macro', '?')}")
print(f"  ep: {ckpt.get('ep', '?')}")
print(f"  Stage 1 v2 parent val_ns22: {ckpt.get('stage1_v2_val_ns22', '?')}")
print(f"\nModel params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")


In [ ]:
# Cell 4: Enumerate test files
if TEST_DIR is None or not TEST_DIR.exists():
    print("⚠ test_soundscapes not mounted, fallback to empty (Save Version mode)")
    test_files = []
else:
    test_files = sorted(TEST_DIR.glob("*.ogg"))
    if not test_files:
        # Try other formats
        test_files = sorted(TEST_DIR.glob("*.wav"))
    print(f"Test files: {len(test_files)}")
    if test_files:
        print(f"  Sample: {test_files[0].name}")


In [ ]:
# Cell 5: Inference functions
@torch.no_grad()
def infer_file(file_path):
    """Read a test file, chunk to 12×5s, predict probabilities.

    Returns: (N_WINDOWS, N_CLASSES) array of sigmoid probabilities.
    """
    try:
        wav, sr = sf.read(str(file_path), dtype="float32")
        if wav.ndim > 1: wav = wav.mean(axis=1)
        if sr != SR:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
    except Exception as e:
        print(f"  [read err] {file_path.name}: {str(e)[:80]}")
        return np.full((N_WINDOWS, N_CLASSES), 0.0, dtype=np.float32)

    # Pad / chunk to N_WINDOWS × CHUNK_LEN
    target_len = N_WINDOWS * CHUNK_LEN
    if len(wav) < target_len:
        wav = np.pad(wav, (0, target_len - len(wav)))
    chunks = wav[:target_len].reshape(N_WINDOWS, CHUNK_LEN)

    # Batch inference
    batch = torch.from_numpy(chunks).float()  # (N_WINDOWS, CHUNK_LEN)
    mel = mel_extractor(batch)                # (N_WINDOWS, n_mels, T)
    logit = model(mel)                        # (N_WINDOWS, N_CLASSES)
    prob = torch.sigmoid(logit).numpy()
    return prob


In [ ]:
# Cell 6: Run inference on all test files
if not test_files:
    print("No test files, skip inference (Save Version mode)")
    all_rows = []
else:
    all_rows = []
    t0 = time.time()
    for fi, fp in enumerate(test_files):
        probs = infer_file(fp)   # (N_WINDOWS, N_CLASSES)
        stem = fp.stem
        for wi in range(N_WINDOWS):
            end_sec = (wi + 1) * 5
            row_id = f"{stem}_{end_sec}"
            row = {"row_id": row_id}
            for ci, lbl in enumerate(PRIMARY_LABELS):
                row[lbl] = float(probs[wi, ci])
            all_rows.append(row)
        if (fi + 1) % 50 == 0 or fi == len(test_files) - 1:
            elapsed = time.time() - t0
            rate = (fi + 1) / elapsed
            eta = (len(test_files) - fi - 1) / rate
            print(f"  [{fi+1}/{len(test_files)}] {rate*60:.1f}/min ETA {eta/60:.1f}min")
    print(f"\nInference done in {(time.time()-t0)/60:.1f}min")


In [ ]:
# Cell 7: TopN N=1 PP (paper 256 spec, exp048 と同)
cols = ["row_id"] + PRIMARY_LABELS
if all_rows:
    sub_df = pd.DataFrame(all_rows)[cols]
else:
    # Save Version: use sample_submission as template
    sub_df = sample.copy()

print(f"Submission shape: {sub_df.shape}")

if all_rows:
    # TopN N=1 PP: per-file × per-class top-K max scaling
    FCS_TOP_K = 1
    FCS_POWER = 1.0

    preds_array = sub_df[PRIMARY_LABELS].values.astype(np.float32)
    _n, _c = preds_array.shape
    _view = preds_array.reshape(-1, N_WINDOWS, _c)
    _sorted = np.sort(_view, axis=1)
    _topk_mean = _sorted[:, -FCS_TOP_K:, :].mean(axis=1, keepdims=True)
    _scale = np.power(_topk_mean, FCS_POWER)
    preds_array = (_view * _scale).reshape(_n, _c).astype(np.float32)

    sub_df[PRIMARY_LABELS] = preds_array
    print(f"TopN N=1 PP applied (K={FCS_TOP_K}, POW={FCS_POWER})")
    print(f"  pred mean: {preds_array.mean():.4f}, max: {preds_array.max():.4f}")


In [ ]:
# Cell 8: Save submission.csv
out_path = "submission.csv"
sub_df.to_csv(out_path, index=False)
print(f"Saved {out_path}")
print(f"  shape: {sub_df.shape}")
print(f"  head:")
print(sub_df.head(3).to_string())
